In [1]:
# User_Interface.ipynb
import ipywidgets as widgets
from IPython.display import display, Image, clear_output
import pandas as pd
import os
import sys
import asyncio
import inspect
from datetime import datetime

# 直接获取当前工作目录（就是项目根目录）
B_ROOT_PATH = os.getcwd()

# 加入系统路径
if B_ROOT_PATH not in sys.path:
    sys.path.append(B_ROOT_PATH)

print(f"项目根目录已设置为: {B_ROOT_PATH}")

# ========== 固定配置（无需修改） ==========
UPLOAD_DIR = os.path.abspath(os.path.join(B_ROOT_PATH, "assets/input"))
os.makedirs(UPLOAD_DIR, exist_ok=True)

# ========== 适配Jupyter事件循环（关键修复） ==========
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    print("⚠️  未安装 nest_asyncio，将自动安装...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nest_asyncio", "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"])
    import nest_asyncio
    nest_asyncio.apply()

# ========== 1. 导入B的工具并做前置检查 ==========
try:
    from tools.hello_tool import detect_image
    from core.utils import draw_boxes, save_result
    
    sig = inspect.signature(detect_image)
    param_names = list(sig.parameters.keys())
    status_text = f"🟢 所有工具加载成功"
    status_color = 'green'
    load_success = True
except ImportError as e:
    error_msg = str(e)
    if "torch" in error_msg:
        status_text = "🔴 缺少torch库！请执行：pip install torch torchvision"
    elif "tools.hello_tool" in error_msg:
        status_text = f"🔴 B路径错误！当前路径：{B_ROOT_PATH}"
    else:
        status_text = f"🔴 工具加载失败：{error_msg}"
    status_color = 'red'
    load_success = False

# ========== 2. 界面控件 ==========
status_label = widgets.Label(
    value=status_text,
    style={'color': status_color, 'font_size': '14px'}
)

uploader = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='📤 上传图片',
    layout={'width': '300px'}
)

model_selector = widgets.Dropdown(
    options=['v5', 'v8'],
    value='v5',
    description='🔧 模型类型：',
    layout={'width': '250px'}
)

conf_slider = widgets.FloatSlider(
    value=0.1,  # 降低置信度，确保A端能检测到目标
    min=0.01,   # 最小值降到0.01，兜底检测
    max=0.9,
    step=0.05,
    description='置信度阈值：',
    layout={'width': '300px'}
)

detect_btn = widgets.Button(
    description='▶️ 执行检测',
    button_style='success' if load_success else 'danger',
    disabled=not load_success,
    layout={'width': '200px'}
)

result_area = widgets.Output(layout={'margin': '10px 0', 'border': '1px solid #eee', 'padding': '10px'})

# ========== 3. 核心检测逻辑（适配A端objects字段+补全坐标） ==========
async def async_detect(img_abs_path, model_type, conf_threshold):
    """异步检测函数，适配Jupyter事件循环"""
    detect_result = await detect_image(
        image_path=img_abs_path,
        model_type=model_type,
        conf_threshold=conf_threshold
    )
    return detect_result

def run_detection(btn):
    with result_area:
        clear_output()
        print("🔍 开始检测...")
        
        if not uploader.value:
            print("⚠️ 请先点击「📤 上传图片」选择图片！")
            return
        
        try:
            # 步骤1：保存图片
            if isinstance(uploader.value, dict):
                uploaded_file = list(uploader.value.values())[0]
                file_content = uploaded_file['content']
            else:
                uploaded_file = uploader.value[0]
                file_content = uploaded_file.content
            
            img_filename = f"detect_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.jpg"
            img_abs_path = os.path.join(UPLOAD_DIR, img_filename)
            
            with open(img_abs_path, 'wb') as f:
                f.write(file_content)
            print(f"✅ 图片已保存：{img_abs_path}")
            
            # 步骤2：在Jupyter中执行异步检测（关键修复）
            print("📡 调用检测工具...")
            if inspect.iscoroutinefunction(detect_image):
                # 在Jupyter中直接await异步函数
                detect_result = asyncio.run(async_detect(
                    img_abs_path,
                    model_selector.value,
                    conf_slider.value
                ))
            else:
                detect_result = detect_image(
                    image_path=img_abs_path,
                    model_type=model_selector.value,
                    conf_threshold=conf_slider.value
                )
            
            # 步骤3：适配结果格式（核心修复：兼容A端objects字段+补全坐标）
            detect_result = detect_result if isinstance(detect_result, dict) else {}

            # 从A端的objects字段提取原始检测结果
            raw_objects = detect_result.get("objects", [])
            standard_items = []

            for obj in raw_objects:
                # 提取并转换字段（适配confidence→conf）
                label = obj.get("label", "未知")
                conf = obj.get("confidence", obj.get("conf", 0.0))  # 兼容confidence/conf两种字段
                bbox = obj.get("bbox", {})
                
                # 补全缺失的坐标字段，适配x1/y1/x2/y2标准格式
                # 自动补全缺失的x1（默认0，或从x_min/left等字段兼容）
                standard_bbox = {
                    "x1": int(bbox.get("x1", bbox.get("x_min", bbox.get("left", 0)))),
                    "y1": int(bbox.get("y1", bbox.get("y_min", bbox.get("top", 0)))),
                    "x2": int(bbox.get("x2", bbox.get("x_max", bbox.get("right", 0)))),
                    "y2": int(bbox.get("y2", bbox.get("y_max", bbox.get("bottom", 0))))
                }
                
                # 过滤无效坐标（避免画框出错）
                if standard_bbox["x1"] >= standard_bbox["x2"]:
                    standard_bbox["x1"] = max(0, standard_bbox["x2"] - 200)  # 兜底x1
                if standard_bbox["y1"] >= standard_bbox["y2"]:
                    standard_bbox["y1"] = max(0, standard_bbox["y2"] - 200)  # 兜底y1
                
                standard_items.append({
                    "label": label,
                    "conf": conf,
                    "bbox": standard_bbox
                })

            # 重新赋值，适配后续的绘图和表格逻辑
            detect_result["items"] = standard_items
            detect_result["detections"] = standard_items
            detect_result["count"] = len(standard_items)

            print(f"✅ 检测完成！共识别 {detect_result['count']} 个目标")
            # 调试日志：打印转换后的标准数据，方便验证
            print(f"📌 A端原始objects：{raw_objects}")
            print(f"🔧 转换后标准数据：{standard_items}")
            
            # 步骤4：生成结果图（加详细报错日志）
            try:
                print(f"🎨 开始绘制检测框，检测到{len(standard_items)}个目标")
                drawn_img = draw_boxes(img_abs_path, detect_result)
                result_img_path = save_result(drawn_img)
                print(f"✅ 带框图片保存：{result_img_path}")
            except Exception as e:
                print(f"⚠️ 结果图生成失败：{str(e)}")
                # 打印完整报错栈，快速定位问题
                import traceback
                traceback.print_exc()
                result_img_path = img_abs_path
            
            # 步骤5：展示结果
            print("\n🖼️ 上传原图：")
            display(Image(img_abs_path, width=400))
            
            print("\n🎯 检测结果图（带框）：")
            display(Image(result_img_path, width=400))
            
            if detect_result["count"] > 0:
                print("\n📋 详细检测结果：")
                df = pd.DataFrame({
                    "目标类别": [item.get("label", "未知") for item in detect_result["items"]],
                    "置信度": [f"{item.get('conf', 0):.2f}" for item in detect_result["items"]],
                    "边界框": [f"[{item.get('bbox', {}).get('x1', 0)}, {item.get('bbox', {}).get('y1', 0)}, {item.get('bbox', {}).get('x2', 0)}, {item.get('bbox', {}).get('y2', 0)}]" 
                             for item in detect_result["items"]]
                })
                display(df)
            else:
                print("\n📋 未检测到任何目标")
        
        except Exception as e:
            print(f"\n❌ 检测失败：{str(e)}")
            print("💡 排查提示：")
            print(f"   1. B路径是否正确：{B_ROOT_PATH}")
            print(f"   2. 是否安装torch：pip install torch torchvision")
            print(f"   3. B的函数参数是否匹配：{param_names}")

# ========== 4. 绑定事件 + 展示界面 ==========
detect_btn.on_click(run_detection)

ui_layout = widgets.VBox([
    status_label,
    uploader,
    model_selector,
    conf_slider,
    detect_btn,
    result_area
], layout={'padding': '15px', 'border': '1px solid #ddd', 'border_radius': '5px'})

display(ui_layout)



项目根目录已设置为: d:\hust\YA_MCPServer_Yolo
